In [1]:
!pip install -q pyyaml

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="Mvk7yWOvu1H3D6DdtENm")

In [3]:
project = rf.workspace("fyp-v3pnw").project("fyp-amjew")
version = project.version(5)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FYP-5 in yolov11:: 100%|██████████| 2892/2892 [00:00<00:00, 5407.07it/s]


In [4]:
project = rf.workspace("balldt").project("football-player-detection-0lklm")
version = project.version(7)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Football-player-detection-7 in yolov11:: 100%|██████████| 2479/2479 [00:00<00:00, 9659.15it/s]


In [5]:
project = rf.workspace("project11-vwk2l").project("x_321_5star")
version = project.version(13)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to X_321_5star-13 in yolov11:: 100%|██████████| 18788/18788 [00:02<00:00, 7555.65it/s] 


In [6]:
project = rf.workspace("smad").project("referee-ogri2")
version = project.version(9)
dataset = version.download("yolov11")                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Referee-9 in yolov11:: 100%|██████████| 18130/18130 [00:01<00:00, 9195.40it/s] 


In [7]:
import os
HOME = os.getcwd()
print(HOME)

/home/ubuntu/projects/sure-football-analysis


In [8]:
import os
import shutil
import yaml
from collections import defaultdict

# --- Configuration ---

# 1. Define your target classes and their order for the new dataset
TARGET_CLASSES = ['ball', 'player', 'goalkeeper', 'referee']
TARGET_CLASS_TO_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}
NEW_NC = len(TARGET_CLASSES)

# 2. Configure your input datasets
#    - 'yaml_path': Full path to the data.yaml file of the dataset.
#    - 'id': A unique short identifier for this dataset (used for prefixing files to avoid name collisions).
#    - 'class_mapping_to_target':
#        A dictionary mapping original class names (lowercase) from this dataset
#        to the desired target class name in TARGET_CLASSES.
#        Use `None` to discard a class.
DATASETS_CONFIG = [
    {
        'id': 'FYP-5',
        'yaml_path': '/home/ubuntu/projects/sure-football-analysis/FYP-5/data.yaml', # IMPORTANT: Update this path
        'original_yaml_content_for_parsing': """
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 4
names: ['ball', 'goalkeeper', 'player', 'referee']
""",
        'class_mapping_to_target': {
            'ball': 'ball',
            'goalkeeper': 'goalkeeper',
            'player': 'player',
            'referee': 'referee',
        }
    },
    {
        'id': 'Football-player-detection-7',
        'yaml_path': '/home/ubuntu/projects/sure-football-analysis/Football-player-detection-7/data.yaml', # IMPORTANT: Update this path
        'original_yaml_content_for_parsing': """
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 5
names: ['football', 'goalie', 'goalkeeper', 'player', 'referee']
""",
        'class_mapping_to_target': {
            'football': 'ball',
            'goalie': 'goalkeeper',
            'goalkeeper': 'goalkeeper', # Merges with 'goalie' into target 'goalkeeper'
            'player': 'player',
            'referee': 'referee',
        }
    },
    {
        'id': 'Referee-9',
        'yaml_path': '/home/ubuntu/projects/sure-football-analysis/Referee-9/data.yaml', # IMPORTANT: Update this path
        'original_yaml_content_for_parsing': """
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 3
names: ['ball', 'player', 'referee']
""",
        'class_mapping_to_target': {
            'ball': 'ball',
            'player': 'player',
            'referee': 'referee',
            # 'goalkeeper' is missing, which is fine.
        }
    },
    {
        'id': 'X_321_5star-13',
        'yaml_path': '/home/ubuntu/projects/sure-football-analysis/X_321_5star-13/data.yaml', # IMPORTANT: Update this path
        'original_yaml_content_for_parsing': """
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 6
names: ['Ball', 'Goalkeeper', 'Main referee', 'Player', 'Side referee', 'Staff members']
""",
        'class_mapping_to_target': {
            'ball': 'ball',             # Handles case difference
            'goalkeeper': 'goalkeeper', # Handles case difference
            'main referee': 'referee',
            'player': 'player',
            'side referee': 'referee',  # Merged into 'referee'
            'staff members': None,      # This class will be discarded
        }
    }
]

# 3. Define output directory for the unified dataset
#    This script will create this directory and subdirectories (images/train, labels/val, etc.)
OUTPUT_DATASET_DIR = '/home/ubuntu/projects/sure-football-analysis/unified_soccer_dataset' # IMPORTANT: Update if needed

In [9]:
# --- Helper Functions ---

def parse_yaml_content(yaml_content_str):
    """Parses YAML content string."""
    try:
        return yaml.safe_load(yaml_content_str)
    except yaml.YAMLError as e:
        print(f"Error parsing YAML content: {e}")
        return None

def process_label_file(original_label_path, new_label_path, original_class_id_to_name, dataset_class_mapping_to_target):
    """
    Reads an original YOLO label file, remaps class IDs, and writes a new label file.
    """
    if not os.path.exists(original_label_path):
        open(new_label_path, 'w').close() # Create empty label file for corresponding image
        return 0

    processed_annotations_count = 0
    with open(original_label_path, 'r') as f_in, open(new_label_path, 'w') as f_out:
        for line in f_in:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                original_class_id = int(parts[0])
                coords = parts[1:]
            except ValueError:
                print(f"Warning: Skipping malformed line in {original_label_path}: {line.strip()}")
                continue

            original_class_name = original_class_id_to_name.get(original_class_id)
            if original_class_name is None:
                continue

            target_class_name = dataset_class_mapping_to_target.get(original_class_name.lower())
            if target_class_name is None:
                continue

            new_class_id = TARGET_CLASS_TO_ID.get(target_class_name)
            if new_class_id is None:
                print(f"Error: Target class '{target_class_name}' not found in TARGET_CLASS_TO_ID. Skipping.")
                continue

            f_out.write(f"{new_class_id} {' '.join(coords)}\n")
            processed_annotations_count += 1
    return processed_annotations_count

In [10]:
print(f"Creating unified dataset at: {OUTPUT_DATASET_DIR}")
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DATASET_DIR, 'labels', split), exist_ok=True)

total_images_processed = 0
total_annotations_processed = 0

for config in DATASETS_CONFIG:
    dataset_id = config['id'] # This is now e.g., 'FYP-5'
    dataset_yaml_path = config['yaml_path']
    dataset_class_mapping = config['class_mapping_to_target']
    print(f"\nProcessing dataset: {dataset_id} (from {dataset_yaml_path})")

    base_dir_for_relative_paths = None
    original_yaml_data = None

    if os.path.exists(dataset_yaml_path):
        try:
            with open(dataset_yaml_path, 'r') as f:
                original_yaml_data = yaml.safe_load(f)
            base_dir_for_relative_paths = os.path.dirname(dataset_yaml_path)
        except Exception as e:
            print(f"Error loading YAML file {dataset_yaml_path}: {e}")
    
    if not original_yaml_data: # Fallback or if file loading failed
        print(f"Warning: Could not load YAML from file {dataset_yaml_path}. Attempting to parse fallback content string for {dataset_id}...")
        original_yaml_data = parse_yaml_content(config['original_yaml_content_for_parsing'])
        if original_yaml_data:
              # If using fallback, base_dir needs to be assumed or explicitly set.
              # For this script, we'll assume it's the directory part of the 'yaml_path' config.
            base_dir_for_relative_paths = os.path.dirname(dataset_yaml_path)
            if not os.path.exists(base_dir_for_relative_paths) and not os.path.isabs(dataset_yaml_path):
                # This case is tricky if yaml_path is not a full path and doesn't exist
                # For /content/FYP-5/data.yaml, dirname is /content/FYP-5
                print(f"Warning: Fallback base directory {base_dir_for_relative_paths} might not be correct if YAML file doesn't exist at specified path.")
        else:
            print(f"ERROR: Could not parse YAML from file or fallback string for {dataset_id}. Skipping.")
            continue
    
    if not original_yaml_data or 'names' not in original_yaml_data:
        print(f"ERROR: 'names' field not found in YAML data for {dataset_id}. Skipping.")
        continue
    
    original_class_id_to_name = {i: name for i, name in enumerate(original_yaml_data['names'])}

    for split in ['train', 'val', 'test']:
        print(f"  Processing split: {split}")
        relative_images_path_from_yaml = original_yaml_data.get(split)
        if not relative_images_path_from_yaml:
            print(f"    Split '{split}' not defined in {dataset_yaml_path}. Skipping.")
            continue

        # --- MODIFIED PATH LOGIC ---
        # Correct the relative path if it starts with '../' or '..\'
        # This assumes that if '..' is present, it's because the YAML incorrectly
        # specifies going up a directory, but the actual structure is flat
        # (e.g., images are in 'dataset_folder/train/images', not 'dataset_folder/../train/images')
        
        corrected_relative_img_path = relative_images_path_from_yaml
        if corrected_relative_img_path.startswith(('../', '..\\')):
            corrected_relative_img_path = corrected_relative_img_path[3:]
        
        original_images_dir = os.path.abspath(os.path.join(base_dir_for_relative_paths, corrected_relative_img_path))
        
        # Derive label path from the corrected image path structure
        corrected_relative_label_path = corrected_relative_img_path.replace('images', 'labels') # More robust than /images
        if 'images' not in corrected_relative_img_path.lower(): # if path is just 'train' not 'train/images'
              corrected_relative_label_path = os.path.join(os.path.dirname(corrected_relative_img_path), 'labels')


        original_labels_dir = os.path.abspath(os.path.join(base_dir_for_relative_paths, corrected_relative_label_path))
        # --- END OF MODIFIED PATH LOGIC ---

        print(f"    Resolved original images directory: {original_images_dir}")
        print(f"    Resolved original labels directory: {original_labels_dir}")

        if not os.path.isdir(original_images_dir):
            print(f"    Warning: Original images directory not found for {dataset_id} split {split}: {original_images_dir}. Skipping this split.")
            continue

        new_images_dir = os.path.join(OUTPUT_DATASET_DIR, 'images', split)
        new_labels_dir = os.path.join(OUTPUT_DATASET_DIR, 'labels', split)

        image_files = [f for f in os.listdir(original_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"    No image files found in {original_images_dir}")
            continue

        for image_filename in image_files:
            original_image_path = os.path.join(original_images_dir, image_filename)
            base, ext = os.path.splitext(image_filename)
            new_image_filename = f"{dataset_id}_{base}{ext}" # Use dataset_id (e.g., FYP-5) as prefix
            new_image_path = os.path.join(new_images_dir, new_image_filename)

            try:
                shutil.copy2(original_image_path, new_image_path)
            except Exception as e:
                print(f"    Error copying image {original_image_path} to {new_image_path}: {e}")
                continue

            label_filename = f"{base}.txt"
            original_label_path = os.path.join(original_labels_dir, label_filename)
            new_label_filename = f"{dataset_id}_{base}.txt"
            new_label_path = os.path.join(new_labels_dir, new_label_filename)

            annotations_in_file = process_label_file(original_label_path, new_label_path, original_class_id_to_name, dataset_class_mapping)
            total_annotations_processed += annotations_in_file
            total_images_processed += 1

print(f"\n--- Processing Complete ---")
print(f"Total images processed and copied: {total_images_processed}")
print(f"Total annotations remapped: {total_annotations_processed}")
print(f"Unified dataset created at: {OUTPUT_DATASET_DIR}")

Creating unified dataset at: /home/ubuntu/projects/sure-football-analysis/unified_soccer_dataset

Processing dataset: FYP-5 (from /home/ubuntu/projects/sure-football-analysis/FYP-5/data.yaml)
  Processing split: train
    Resolved original images directory: /home/ubuntu/projects/sure-football-analysis/FYP-5/train/images
    Resolved original labels directory: /home/ubuntu/projects/sure-football-analysis/FYP-5/train/labels
  Processing split: val
    Resolved original images directory: /home/ubuntu/projects/sure-football-analysis/FYP-5/valid/images
    Resolved original labels directory: /home/ubuntu/projects/sure-football-analysis/FYP-5/valid/labels
  Processing split: test
    Resolved original images directory: /home/ubuntu/projects/sure-football-analysis/FYP-5/test/images
    Resolved original labels directory: /home/ubuntu/projects/sure-football-analysis/FYP-5/test/labels

Processing dataset: Football-player-detection-7 (from /home/ubuntu/projects/sure-football-analysis/Football-pl

In [11]:
unified_yaml_data = {
    'path': os.path.abspath(OUTPUT_DATASET_DIR),
    'train': '../images/train',
    'val': '../images/val',
    'test': '../images/test',
    'nc': NEW_NC,
    'names': TARGET_CLASSES,
    'roboflow': {
        'workspace': 'custom-merged',
        'project': 'unified-soccer-dataset',
        'version': 1,
        'license': 'custom',
        'url': 'local'
    }
}
unified_yaml_path = os.path.join(OUTPUT_DATASET_DIR, 'data_unified.yaml')
with open(unified_yaml_path, 'w') as f:
    yaml.dump(unified_yaml_data, f, sort_keys=False, default_flow_style=False)
print(f"Unified data.yaml created at: {unified_yaml_path}")
print("\nContent of the new data_unified.yaml:")
print(yaml.dump(unified_yaml_data, sort_keys=False, default_flow_style=False))

Unified data.yaml created at: /home/ubuntu/projects/sure-football-analysis/unified_soccer_dataset/data_unified.yaml

Content of the new data_unified.yaml:
path: /home/ubuntu/projects/sure-football-analysis/unified_soccer_dataset
train: ../images/train
val: ../images/val
test: ../images/test
nc: 4
names:
- ball
- player
- goalkeeper
- referee
roboflow:
  workspace: custom-merged
  project: unified-soccer-dataset
  version: 1
  license: custom
  url: local

